<a href="https://colab.research.google.com/github/akul-bharadwaj/various-agents/blob/main/Case_Study_3_LangGraph_Trading_Agent_Termination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 3: Termination and Infinite-Loop Prevention
## LangGraph Trading Strategy Agent

### Objective

Build a cyclic trading workflow that repeatedly:

1. fetches mock market data;
2. asks an OpenAI model to recalculate a strategy;
3. evaluates whether execution should continue; and
4. terminates safely.

The required termination conditions are:

- **maximum iterations**;
- **confidence threshold**.

Additional safeguards are included:

- no-progress detection;
- repeated API-error detection;
- LangGraph's runtime `recursion_limit`;
- a deliberately broken graph to demonstrate infinite-loop behaviour.

> **Educational disclaimer:** This notebook uses fictional market data and simplified logic. It is not financial or trading advice and must not be used for real trades.

## Why LangGraph?

This problem is naturally represented as a graph containing a cycle:

```text
START
  │
  ▼
Fetch Market Data
  │
  ▼
Recalculate Strategy with OpenAI
  │
  ▼
Evaluate Termination
  │
  ├── Confidence reached ──────────► END
  │
  ├── Maximum iterations reached ─► END
  │
  ├── No progress detected ───────► END
  │
  ├── Repeated API errors ────────► END
  │
  └── Continue ───────────────────► Fetch Market Data
```

LangGraph is useful here because it provides explicit graph state, cyclic edges, conditional routing to `END`, and a graph-level recursion limit as a final safety mechanism.

## 1. Install dependencies

In [1]:
!pip -q install --upgrade langgraph langchain-openai pydantic pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


## 2. Configure the OpenAI API key

The key is entered with `getpass`, so it is not printed or hard-coded in the notebook.

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured.")

Enter your OpenAI API key: ··········
OpenAI API key configured.


## 3. Imports and package versions

In [3]:
import copy
import json
import logging
import operator
from importlib.metadata import version
from pprint import pprint
from typing import Annotated, Any, Dict, List, Literal, TypedDict

import pandas as pd
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)
logger = logging.getLogger("trading_agent")

print("Model:", MODEL)
print("langgraph:", version("langgraph"))
print("langchain-openai:", version("langchain-openai"))
print("pydantic:", version("pydantic"))

Model: gpt-4o-mini
langgraph: 1.2.9
langchain-openai: 1.3.5
pydantic: 2.13.4


## 4. Define structured OpenAI output

The model proposes:

- a strategy;
- a short rationale; and
- its own raw confidence.

The workflow separately computes a deterministic **calibrated confidence** from market signals. This keeps the termination demonstrations reproducible instead of depending entirely on variable model output.

In [4]:
class StrategyProposal(BaseModel):
    """Structured strategy returned by the OpenAI model."""

    strategy: Literal["BUY", "HOLD", "SELL", "HEDGE"]
    rationale: str = Field(
        min_length=10,
        max_length=500,
    )
    raw_confidence: float = Field(
        ge=0.0,
        le=1.0,
    )


llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
)

strategy_model = llm.with_structured_output(
    StrategyProposal
)

print("Structured strategy model created.")

Structured strategy model created.


## 5. Define graph state

The state persists information across every pass through the cycle.

`history` uses a reducer (`operator.add`) so each node can append trace entries without overwriting earlier entries.

In [5]:
class TradingState(TypedDict):
    """State shared by every node in the trading graph."""

    scenario: str
    market_data: Dict[str, Any]

    strategy: str
    rationale: str
    raw_llm_confidence: float
    confidence: float
    previous_confidence: float

    iteration: int
    max_iterations: int
    confidence_threshold: float

    stagnant_iterations: int
    max_stagnant_iterations: int

    api_error_count: int
    max_api_errors: int

    terminated: bool
    termination_reason: str

    history: Annotated[
        List[Dict[str, Any]],
        operator.add,
    ]


def create_initial_state(
    scenario: str,
    max_iterations: int,
    confidence_threshold: float,
    max_stagnant_iterations: int = 4,
    max_api_errors: int = 2,
) -> TradingState:
    """Create a fresh state for one graph execution."""
    return {
        "scenario": scenario,
        "market_data": {},
        "strategy": "NOT_CALCULATED",
        "rationale": "",
        "raw_llm_confidence": 0.0,
        "confidence": 0.0,
        "previous_confidence": 0.0,
        "iteration": 0,
        "max_iterations": max_iterations,
        "confidence_threshold": confidence_threshold,
        "stagnant_iterations": 0,
        "max_stagnant_iterations": max_stagnant_iterations,
        "api_error_count": 0,
        "max_api_errors": max_api_errors,
        "terminated": False,
        "termination_reason": "",
        "history": [],
    }


pprint(
    create_initial_state(
        scenario="trending",
        max_iterations=6,
        confidence_threshold=0.80,
    )
)

{'api_error_count': 0,
 'confidence': 0.0,
 'confidence_threshold': 0.8,
 'history': [],
 'iteration': 0,
 'market_data': {},
 'max_api_errors': 2,
 'max_iterations': 6,
 'max_stagnant_iterations': 4,
 'previous_confidence': 0.0,
 'rationale': '',
 'raw_llm_confidence': 0.0,
 'scenario': 'trending',
 'stagnant_iterations': 0,
 'strategy': 'NOT_CALCULATED',
 'terminated': False,
 'termination_reason': ''}


## 6. Create deterministic mock market-data streams

Two scenarios are provided:

### `trending`

The signals become progressively clearer. Calibrated confidence should eventually cross the threshold.

### `choppy`

Signals remain weak and volatile. Confidence should stay below the threshold, causing termination through `max_iterations`.

The data is fictional and repeatable.

In [6]:
MARKET_SCENARIOS: Dict[str, List[Dict[str, float]]] = {
    "trending": [
        {
            "price": 100.00,
            "daily_return": 0.002,
            "volatility": 0.30,
            "trend_strength": 0.30,
            "momentum": 0.10,
            "signal_agreement": 0.40,
        },
        {
            "price": 101.00,
            "daily_return": 0.010,
            "volatility": 0.24,
            "trend_strength": 0.55,
            "momentum": 0.50,
            "signal_agreement": 0.65,
        },
        {
            "price": 102.82,
            "daily_return": 0.018,
            "volatility": 0.15,
            "trend_strength": 0.82,
            "momentum": 0.78,
            "signal_agreement": 0.90,
        },
        {
            "price": 104.67,
            "daily_return": 0.018,
            "volatility": 0.13,
            "trend_strength": 0.88,
            "momentum": 0.84,
            "signal_agreement": 0.94,
        },
    ],
    "choppy": [
        {
            "price": 100.00,
            "daily_return": 0.001,
            "volatility": 0.55,
            "trend_strength": 0.15,
            "momentum": 0.05,
            "signal_agreement": 0.35,
        },
        {
            "price": 99.70,
            "daily_return": -0.003,
            "volatility": 0.60,
            "trend_strength": 0.20,
            "momentum": -0.10,
            "signal_agreement": 0.30,
        },
        {
            "price": 100.10,
            "daily_return": 0.004,
            "volatility": 0.50,
            "trend_strength": 0.10,
            "momentum": 0.08,
            "signal_agreement": 0.40,
        },
        {
            "price": 99.90,
            "daily_return": -0.002,
            "volatility": 0.58,
            "trend_strength": 0.18,
            "momentum": -0.04,
            "signal_agreement": 0.32,
        },
    ],
}


def get_mock_market_data(
    scenario: str,
    iteration: int,
) -> Dict[str, Any]:
    """Return repeatable fictional market data."""
    if scenario not in MARKET_SCENARIOS:
        raise ValueError(
            f"Unknown scenario: {scenario!r}. "
            f"Choose from {list(MARKET_SCENARIOS)}."
        )

    data_stream = MARKET_SCENARIOS[scenario]
    index = iteration % len(data_stream)

    return {
        **copy.deepcopy(data_stream[index]),
        "scenario": scenario,
        "sample_index": index,
    }


pprint(get_mock_market_data("trending", 0))

{'daily_return': 0.002,
 'momentum': 0.1,
 'price': 100.0,
 'sample_index': 0,
 'scenario': 'trending',
 'signal_agreement': 0.4,
 'trend_strength': 0.3,
 'volatility': 0.3}


## 7. Implement the graph nodes

### Confidence calibration

The workflow computes:

```text
confidence =
    0.45
    + 0.35 × trend_strength
    + 0.20 × signal_agreement
    - 0.20 × volatility
```

The result is clipped to `[0, 1]`.

This score is deliberately transparent and simplified for the case study.

In [7]:
def calculate_calibrated_confidence(
    market_data: Dict[str, Any],
) -> float:
    """Calculate a deterministic confidence score."""
    score = (
        0.45
        + 0.35 * market_data["trend_strength"]
        + 0.20 * market_data["signal_agreement"]
        - 0.20 * market_data["volatility"]
    )

    return round(
        max(0.0, min(1.0, score)),
        4,
    )


def fetch_market_data_node(
    state: TradingState,
) -> Dict[str, Any]:
    """Fetch the next fictional market snapshot."""
    market_data = get_mock_market_data(
        scenario=state["scenario"],
        iteration=state["iteration"],
    )

    logger.info(
        "FETCH | iteration=%d | price=%.2f | scenario=%s",
        state["iteration"] + 1,
        market_data["price"],
        state["scenario"],
    )

    return {
        "market_data": market_data,
        "history": [
            {
                "event": "MARKET_DATA_FETCHED",
                "iteration": state["iteration"] + 1,
                "market_data": market_data,
            }
        ],
    }


def recalculate_strategy_node(
    state: TradingState,
) -> Dict[str, Any]:
    """Ask OpenAI to recalculate the strategy."""
    prompt = f"""
You are analysing fictional market data for an educational
agent-loop demonstration.

Return a cautious strategy proposal using only the supplied data.
Do not claim that this is financial advice.

Current market data:
{json.dumps(state["market_data"], indent=2)}

Previous strategy: {state["strategy"]}
Previous calibrated confidence: {state["confidence"]}

Select one strategy from BUY, HOLD, SELL, or HEDGE.
Keep the rationale brief and grounded in the market fields.
"""

    try:
        proposal = strategy_model.invoke(prompt)
        next_api_error_count = 0
        model_error = None

    except Exception as exc:
        proposal = StrategyProposal(
            strategy="HOLD",
            rationale=(
                "The model call failed, so the workflow uses a "
                "conservative fallback and waits for human review."
            ),
            raw_confidence=0.0,
        )
        next_api_error_count = state["api_error_count"] + 1
        model_error = f"{type(exc).__name__}: {exc}"

    calibrated_confidence = calculate_calibrated_confidence(
        state["market_data"]
    )

    confidence_change = abs(
        calibrated_confidence - state["confidence"]
    )

    if (
        state["iteration"] > 0
        and confidence_change < 0.005
    ):
        stagnant_iterations = (
            state["stagnant_iterations"] + 1
        )
    else:
        stagnant_iterations = 0

    next_iteration = state["iteration"] + 1

    logger.info(
        "STRATEGY | iteration=%d | strategy=%s | "
        "confidence=%.4f",
        next_iteration,
        proposal.strategy,
        calibrated_confidence,
    )

    trace_entry = {
        "event": "STRATEGY_RECALCULATED",
        "iteration": next_iteration,
        "strategy": proposal.strategy,
        "rationale": proposal.rationale,
        "raw_llm_confidence": proposal.raw_confidence,
        "calibrated_confidence": calibrated_confidence,
        "confidence_change": round(
            confidence_change,
            4,
        ),
        "api_error": model_error,
    }

    return {
        "strategy": proposal.strategy,
        "rationale": proposal.rationale,
        "raw_llm_confidence": proposal.raw_confidence,
        "previous_confidence": state["confidence"],
        "confidence": calibrated_confidence,
        "iteration": next_iteration,
        "stagnant_iterations": stagnant_iterations,
        "api_error_count": next_api_error_count,
        "history": [trace_entry],
    }


def evaluate_termination_node(
    state: TradingState,
) -> Dict[str, Any]:
    """Evaluate all explicit termination and safety rules."""
    terminated = False
    reason = "CONTINUE"

    if state["api_error_count"] >= state["max_api_errors"]:
        terminated = True
        reason = "MAX_API_ERRORS_REACHED"

    elif state["confidence"] >= state["confidence_threshold"]:
        terminated = True
        reason = "CONFIDENCE_THRESHOLD_REACHED"

    elif state["iteration"] >= state["max_iterations"]:
        terminated = True
        reason = "MAX_ITERATIONS_REACHED"

    elif (
        state["stagnant_iterations"]
        >= state["max_stagnant_iterations"]
    ):
        terminated = True
        reason = "NO_PROGRESS_DETECTED"

    logger.info(
        "EVALUATE | iteration=%d | terminated=%s | reason=%s",
        state["iteration"],
        terminated,
        reason,
    )

    return {
        "terminated": terminated,
        "termination_reason": (
            reason if terminated else ""
        ),
        "history": [
            {
                "event": "TERMINATION_EVALUATED",
                "iteration": state["iteration"],
                "terminated": terminated,
                "reason": reason,
                "confidence": state["confidence"],
            }
        ],
    }


def route_after_evaluation(
    state: TradingState,
) -> Literal["continue", "end"]:
    """Route either back into the loop or to END."""
    return "end" if state["terminated"] else "continue"


print("Graph nodes defined.")

Graph nodes defined.


## 8. Build the safe cyclic graph

The graph contains an intentional cycle, but `evaluate_termination` can route it to `END`.

The runtime recursion limit remains independent of the business stop rules.

In [8]:
safe_builder = StateGraph(TradingState)

safe_builder.add_node(
    "fetch_market_data",
    fetch_market_data_node,
)
safe_builder.add_node(
    "recalculate_strategy",
    recalculate_strategy_node,
)
safe_builder.add_node(
    "evaluate_termination",
    evaluate_termination_node,
)

safe_builder.add_edge(
    START,
    "fetch_market_data",
)
safe_builder.add_edge(
    "fetch_market_data",
    "recalculate_strategy",
)
safe_builder.add_edge(
    "recalculate_strategy",
    "evaluate_termination",
)

safe_builder.add_conditional_edges(
    "evaluate_termination",
    route_after_evaluation,
    {
        "continue": "fetch_market_data",
        "end": END,
    },
)

safe_trading_graph = safe_builder.compile()

print("Safe trading graph compiled.")

Safe trading graph compiled.


## 9. Helper for running the safe graph

`recursion_limit` is passed as a top-level runtime configuration key.

It should be set comfortably above the number of graph steps expected from the explicit `max_iterations` rule.

In [9]:
def run_safe_trading_agent(
    scenario: str,
    max_iterations: int,
    confidence_threshold: float,
    recursion_limit: int = 50,
    max_stagnant_iterations: int = 4,
    max_api_errors: int = 2,
) -> TradingState:
    """Invoke the safe graph and return its final state."""
    initial_state = create_initial_state(
        scenario=scenario,
        max_iterations=max_iterations,
        confidence_threshold=confidence_threshold,
        max_stagnant_iterations=max_stagnant_iterations,
        max_api_errors=max_api_errors,
    )

    final_state = safe_trading_graph.invoke(
        initial_state,
        config={
            "recursion_limit": recursion_limit,
        },
    )

    print("\nFinal safe-graph result")
    print("-" * 60)
    print("Scenario:", final_state["scenario"])
    print("Iterations:", final_state["iteration"])
    print("Strategy:", final_state["strategy"])
    print("Confidence:", final_state["confidence"])
    print(
        "Threshold:",
        final_state["confidence_threshold"],
    )
    print(
        "Termination reason:",
        final_state["termination_reason"],
    )

    return final_state

## 10. Test A — Termination through confidence threshold

The `trending` market stream becomes clearer over successive iterations. With a threshold of `0.80`, the graph should terminate before the maximum iteration count.

In [10]:
confidence_result = run_safe_trading_agent(
    scenario="trending",
    max_iterations=8,
    confidence_threshold=0.80,
    recursion_limit=50,
)

assert confidence_result["terminated"] is True
assert (
    confidence_result["termination_reason"]
    in {
        "CONFIDENCE_THRESHOLD_REACHED",
        "MAX_API_ERRORS_REACHED",
    }
)

pprint(
    {
        "iterations": confidence_result["iteration"],
        "strategy": confidence_result["strategy"],
        "confidence": confidence_result["confidence"],
        "termination_reason": confidence_result[
            "termination_reason"
        ],
    }
)


Final safe-graph result
------------------------------------------------------------
Scenario: trending
Iterations: 3
Strategy: HOLD
Confidence: 0.887
Threshold: 0.8
Termination reason: CONFIDENCE_THRESHOLD_REACHED
{'confidence': 0.887,
 'iterations': 3,
 'strategy': 'HOLD',
 'termination_reason': 'CONFIDENCE_THRESHOLD_REACHED'}


## 11. Test B — Termination through maximum iterations

The `choppy` stream keeps confidence low. A high threshold of `0.95` is intentionally difficult to reach, so the graph should terminate at four iterations.

The no-progress limit is set high so it does not pre-empt the required maximum-iteration demonstration.

In [11]:
max_iteration_result = run_safe_trading_agent(
    scenario="choppy",
    max_iterations=4,
    confidence_threshold=0.95,
    recursion_limit=50,
    max_stagnant_iterations=99,
)

assert max_iteration_result["terminated"] is True
assert (
    max_iteration_result["termination_reason"]
    in {
        "MAX_ITERATIONS_REACHED",
        "MAX_API_ERRORS_REACHED",
    }
)

pprint(
    {
        "iterations": max_iteration_result["iteration"],
        "strategy": max_iteration_result["strategy"],
        "confidence": max_iteration_result["confidence"],
        "termination_reason": max_iteration_result[
            "termination_reason"
        ],
    }
)


Final safe-graph result
------------------------------------------------------------
Scenario: choppy
Iterations: 4
Strategy: HEDGE
Confidence: 0.461
Threshold: 0.95
Termination reason: MAX_ITERATIONS_REACHED
{'confidence': 0.461,
 'iterations': 4,
 'strategy': 'HEDGE',
 'termination_reason': 'MAX_ITERATIONS_REACHED'}


## 12. Inspect the safe graph's execution history

Each loop produces trace entries for market-data retrieval, strategy recalculation, and termination evaluation.

In [12]:
safe_history_table = pd.DataFrame(
    confidence_result["history"]
)

display(
    safe_history_table[
        [
            column
            for column in [
                "iteration",
                "event",
                "strategy",
                "calibrated_confidence",
                "terminated",
                "reason",
                "api_error",
            ]
            if column in safe_history_table.columns
        ]
    ]
)

,iteration,event,strategy,calibrated_confidence,terminated,reason,api_error
0,1,MARKET_DATA_FETCHED,NaN,NaN,NaN,NaN,NaN
1,1,STRATEGY_RECALCULATED,HOLD,0.5750,NaN,NaN,NaN
2,1,TERMINATION_EVALUATED,NaN,NaN,False,CONTINUE,NaN
3,2,MARKET_DATA_FETCHED,NaN,NaN,NaN,NaN,NaN
4,2,STRATEGY_RECALCULATED,HEDGE,0.7245,NaN,NaN,NaN
5,2,TERMINATION_EVALUATED,NaN,NaN,False,CONTINUE,NaN
6,3,MARKET_DATA_FETCHED,NaN,NaN,NaN,NaN,NaN
7,3,STRATEGY_RECALCULATED,HOLD,0.8870,NaN,NaN,NaN
8,3,TERMINATION_EVALUATED,NaN,NaN,True,CONFIDENCE_THRESHOLD_REACHED,NaN


# Failure Simulation: Graph Without Termination

The following graph deliberately omits:

- an `END` route;
- a confidence check;
- a maximum-iteration check;
- no-progress detection.

Its final edge always routes back to `fetch_market_data`.

This graph would continue indefinitely without an external safeguard. LangGraph's `recursion_limit` interrupts it by raising `GraphRecursionError`.

In [13]:
broken_builder = StateGraph(TradingState)

broken_builder.add_node(
    "fetch_market_data",
    fetch_market_data_node,
)
broken_builder.add_node(
    "recalculate_strategy",
    recalculate_strategy_node,
)

broken_builder.add_edge(
    START,
    "fetch_market_data",
)
broken_builder.add_edge(
    "fetch_market_data",
    "recalculate_strategy",
)

# Intentional bug: unconditional cycle with no path to END.
broken_builder.add_edge(
    "recalculate_strategy",
    "fetch_market_data",
)

broken_trading_graph = broken_builder.compile()

print("Broken graph compiled intentionally.")

Broken graph compiled intentionally.


## 13. Run and catch the broken graph safely

A deliberately small recursion limit is used to avoid excessive API calls.

The last emitted state is retained so that the partial result can be compared with the properly terminated runs.

In [14]:
def run_broken_trading_agent(
    recursion_limit: int = 8,
) -> Dict[str, Any]:
    """Run the broken graph until LangGraph interrupts it."""
    initial_state = create_initial_state(
        scenario="trending",
        max_iterations=999,
        confidence_threshold=0.999,
        max_stagnant_iterations=999,
        max_api_errors=999,
    )

    last_state: TradingState = copy.deepcopy(
        initial_state
    )
    snapshots_seen = 0
    caught_error = None

    try:
        for snapshot in broken_trading_graph.stream(
            initial_state,
            config={
                "recursion_limit": recursion_limit,
            },
            stream_mode="values",
        ):
            last_state = snapshot
            snapshots_seen += 1

    except GraphRecursionError as exc:
        caught_error = exc
        print("\nGraphRecursionError caught successfully.")
        print(type(exc).__name__ + ":", str(exc))

    if caught_error is None:
        raise RuntimeError(
            "The broken graph unexpectedly finished."
        )

    result = {
        "status": "INTERRUPTED_BY_RECURSION_LIMIT",
        "iterations_completed": last_state["iteration"],
        "last_strategy": last_state["strategy"],
        "last_confidence": last_state["confidence"],
        "explicit_termination_reason": (
            last_state["termination_reason"] or None
        ),
        "snapshots_seen": snapshots_seen,
        "recursion_limit": recursion_limit,
        "error_type": type(caught_error).__name__,
    }

    print("\nPartial broken-graph result:")
    pprint(result)

    return result


broken_result = run_broken_trading_agent(
    recursion_limit=8,
)


GraphRecursionError caught successfully.
GraphRecursionError: Recursion limit of 8 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

Partial broken-graph result:
{'error_type': 'GraphRecursionError',
 'explicit_termination_reason': None,
 'iterations_completed': 4,
 'last_confidence': 0.92,
 'last_strategy': 'HEDGE',
 'recursion_limit': 8,
 'snapshots_seen': 9,
 'status': 'INTERRUPTED_BY_RECURSION_LIMIT'}


## 14. Compare outputs with and without termination

The safe graph returns a valid final state and a meaningful termination reason.

The broken graph never satisfies a business termination condition. It only stops because the graph runtime forcibly interrupts it.

In [15]:
comparison_table = pd.DataFrame(
    [
        {
            "Run": "Safe: confidence threshold",
            "Completed normally": True,
            "Iterations": confidence_result[
                "iteration"
            ],
            "Last confidence": confidence_result[
                "confidence"
            ],
            "Termination reason": confidence_result[
                "termination_reason"
            ],
            "Runtime error": None,
        },
        {
            "Run": "Safe: maximum iterations",
            "Completed normally": True,
            "Iterations": max_iteration_result[
                "iteration"
            ],
            "Last confidence": max_iteration_result[
                "confidence"
            ],
            "Termination reason": max_iteration_result[
                "termination_reason"
            ],
            "Runtime error": None,
        },
        {
            "Run": "Broken: no termination logic",
            "Completed normally": False,
            "Iterations": broken_result[
                "iterations_completed"
            ],
            "Last confidence": broken_result[
                "last_confidence"
            ],
            "Termination reason": (
                "No business termination condition"
            ),
            "Runtime error": broken_result[
                "error_type"
            ],
        },
    ]
)

display(comparison_table)

,Run,Completed normally,Iterations,Last confidence,Termination reason,Runtime error
0,Safe: confidence threshold,True,3,0.887,CONFIDENCE_THRESHOLD_REACHED,NaN
1,Safe: maximum iterations,True,4,0.461,MAX_ITERATIONS_REACHED,NaN
2,Broken: no termination logic,False,4,0.920,No business termination condition,GraphRecursionError


## Expected comparison

| Version | Behaviour |
|---|---|
| Safe — confidence threshold | Ends when confidence reaches the configured threshold |
| Safe — maximum iterations | Ends even when confidence never becomes sufficiently high |
| Broken — no termination logic | Repeats until `GraphRecursionError` interrupts execution |

The graph-level recursion limit is a **backstop**, not a substitute for business-level termination logic.

## 15. Optional no-progress demonstration

The graph also supports a no-progress safeguard. This protects against a loop in which confidence remains effectively unchanged.

A threshold above `1.0` is used so confidence cannot end the workflow first.

In [16]:
no_progress_result = run_safe_trading_agent(
    scenario="choppy",
    max_iterations=20,
    confidence_threshold=1.01,
    recursion_limit=80,
    max_stagnant_iterations=2,
)

print(
    "No-progress demonstration ended through:",
    no_progress_result["termination_reason"],
)

assert no_progress_result["terminated"] is True


Final safe-graph result
------------------------------------------------------------
Scenario: choppy
Iterations: 5
Strategy: HOLD
Confidence: 0.4625
Threshold: 1.01
Termination reason: NO_PROGRESS_DETECTED
No-progress demonstration ended through: NO_PROGRESS_DETECTED


## Safeguards implemented

| Safeguard | Purpose |
|---|---|
| `confidence_threshold` | Ends when the strategy is sufficiently confident |
| `max_iterations` | Guarantees a finite number of business iterations |
| `max_stagnant_iterations` | Stops when confidence ceases to improve |
| `max_api_errors` | Stops repeated external-model failures |
| `recursion_limit` | Runtime backstop for an incorrectly wired graph |
| Structured output | Validates strategy, rationale, and raw confidence |
| Deterministic confidence | Makes test outcomes reproducible |
| Execution history | Makes every state transition inspectable |

## Coding requirements satisfied

| Requirement | Implementation |
|---|---|
| Fetch market data continuously | `fetch_market_data_node()` in a cycle |
| Recalculate strategies | `recalculate_strategy_node()` with OpenAI |
| Maximum-iteration termination | `MAX_ITERATIONS_REACHED` |
| Confidence termination | `CONFIDENCE_THRESHOLD_REACHED` |
| Infinite-loop safeguard | LangGraph `recursion_limit` |
| Failure without termination | `broken_trading_graph` |
| Compare both outputs | `comparison_table` |
| LangChain ecosystem | `langchain-openai` + LangGraph |
| State management | `TradingState` |
| Traceability | Append-only `history` reducer |

## Main learning

Loops are useful in agentic systems, but every loop needs:

1. an explicit success condition;
2. a hard iteration budget;
3. runtime protection against incorrect graph logic; and
4. enough state and logging to explain why execution continued or stopped.

## References

- LangGraph Graph API:  
  https://docs.langchain.com/oss/python/langgraph/graph-api

- LangGraph Graph API usage and loops:  
  https://docs.langchain.com/oss/python/langgraph/use-graph-api

- LangGraph recursion-limit error:  
  https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

- LangChain OpenAI chat integration:  
  https://docs.langchain.com/oss/python/integrations/chat/openai

- OpenAI GPT-4o mini model:  
  https://developers.openai.com/api/docs/models/gpt-4o-mini

- OpenAI structured outputs:  
  https://developers.openai.com/api/docs/guides/structured-outputs